# Nexus-LOB Quant Research Layer Walkthrough
**Microstructure Research, Order-Level Queue Dynamics & Execution Realism**

This walkthrough demonstrates the end-to-end quantitative research spine developed for Nexus-LOB:
1. **ITCH Message Stream**: Normalized order-level message ingestion (`NormalizedEvent`).
2. **FIFO Queue Dynamics**: `OrderLevelTracker` tracking exact price-level queue position (`ahead`, `size`, time priority).
3. **Microstructure Features**: Leak-free LOB imbalance and microprice computed over `StubOrderBook.view()`.
4. **Passive Fill Hazard**: Kaplan–Meier fill survival estimation (`fill_prob_survival`) with cancellations as competing risk.
5. **Execution Simulation**: `OrderBookEnv` with deterministic FIFO queue tracking (`queue_model='fifo'`).

> *Architectural boundary:* Offline historical ITCH research uses `OrderLevelTracker` and `queue_dynamics.py` on message feeds; live reinforcement learning execution simulation in `OrderBookEnv` uses a deterministic price-level queue-ahead tracking model.

In [1]:
import sys
from pathlib import Path
import numpy as np

# Ensure python_quant is on sys.path
ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(ROOT / "python_quant") not in sys.path:
    sys.path.insert(0, str(ROOT / "python_quant"))

from nexus_quant.book_state import Side, StubOrderBook
from nexus_quant.itch_parser import EventType, NormalizedEvent
from nexus_quant.research.queue_dynamics import OrderLevelTracker, fill_prob_survival, censor_open_orders
from nexus_quant.research.features import lob_imbalance, microprice
from nexus_quant.envs.order_book_env import OrderBookEnv

print("Nexus-LOB quant research packages loaded successfully.")

Nexus-LOB quant research packages loaded successfully.


## 1. Synthetic Tape Generation & Order Book Reconstruction
We construct a deterministic sequence of ITCH order additions, executions, and cancels.

In [2]:
events = []
ts = 34_200_000_000_000  # 09:30:00 EST in ns

# Add initial resting liquidity at bid and ask
events.append(NormalizedEvent(EventType.ADD, ts + 10, 1, Side.Bid, 15000, 100))
events.append(NormalizedEvent(EventType.ADD, ts + 20, 2, Side.Bid, 15000, 50))
events.append(NormalizedEvent(EventType.ADD, ts + 30, 3, Side.Ask, 15002, 100))
events.append(NormalizedEvent(EventType.ADD, ts + 40, 4, Side.Ask, 15002, 200))

# Aggressive execution consuming front of bid queue (Order 1)
events.append(NormalizedEvent(EventType.EXECUTE, ts + 50, 1, Side.Bid, 15000, 100))

print(f"Constructed {len(events)} sample ITCH events.")

Constructed 5 sample ITCH events.


## 2. Order-Level FIFO Queue Tracking (`OrderLevelTracker`)
We feed the normalized events into `OrderLevelTracker.on_event()` and verify FIFO queue positions.

In [3]:
tracker = OrderLevelTracker()
for ev in events:
    tracker.on_event(ev)

print(f"Active resting orders: {len(tracker.orders)} | Completed orders: {len(tracker.completed)}")
o1_comp = tracker.completed[0]
o2_rest = tracker.orders[2]
print(f"Order 1 (filled): size0={o1_comp.size0}, filled={o1_comp.filled}, outcome={o1_comp.outcome}")
print(f"Order 2 (resting): size0={o2_rest.size0}, remaining={o2_rest.size}, ahead={o2_rest.ahead}")
print(f"Total bid depth at 15000: {tracker.level_size(Side.Bid, 15000)} shares")

Active resting orders: 3 | Completed orders: 1
Order 1 (filled): size0=100, filled=100, outcome=filled
Order 2 (resting): size0=50, remaining=50, ahead=0
Total bid depth at 15000: 50 shares


## 3. Microstructure Features & Signal Evaluation
We reconstruct the book view using `StubOrderBook` and compute LOB imbalance and microprice.

In [4]:
book = StubOrderBook()
book.add(Side.Bid, 15000, 90)
book.add(Side.Ask, 15002, 300)
view = book.view()

imb = lob_imbalance(view)
mp = microprice(view)
mid = (view["bid_px"][0] + view["ask_px"][0]) / 2.0

print(f"Best Bid: {view['bid_px'][0]} ({view['bid_sz'][0]} shares)")
print(f"Best Ask: {view['ask_px'][0]} ({view['ask_sz'][0]} shares)")
print(f"Midpoint: {mid:.1f} | Microprice: {mp:.2f} | L1 Imbalance: {imb:+.4f}")

Best Bid: 15000 (90 shares)
Best Ask: 15002 (300 shares)
Midpoint: 15001.0 | Microprice: 15000.46 | L1 Imbalance: -0.5385


## 4. Passive Order Fill Hazard via Kaplan–Meier Survival
We compute the empirical Kaplan–Meier survival curve and fill probability P(fill by tau) across horizons.

In [5]:
all_orders = list(tracker.completed) + censor_open_orders(tracker, ts + 100)
horizons = [1.0, 5.0, 10.0, 50.0]
surv = fill_prob_survival(all_orders, horizons=horizons)

for tau, pf, s in zip(surv["tau"], surv["p_fill"], surv["survival"]):
    print(f"tau = {tau:>4.0f} events | P(fill) = {pf:.4f} | Survival = {s:.4f}")
print(f"Total analyzed: {surv['n']} | Fills: {surv['n_fill']} | Censored/Cancelled: {surv['n_cancel']}")

tau =    1 events | P(fill) = 0.0000 | Survival = 1.0000
tau =    5 events | P(fill) = 0.5000 | Survival = 0.5000
tau =   10 events | P(fill) = 0.5000 | Survival = 0.5000
tau =   50 events | P(fill) = 0.5000 | Survival = 0.5000
Total analyzed: 4 | Fills: 1 | Censored/Cancelled: 3


## 5. Execution Simulation with Deterministic FIFO Queue Dynamics (`OrderBookEnv`)
We simulate parent-order execution under `queue_model='fifo'` and inspect the inventory trajectory.

In [6]:
env = OrderBookEnv(seed=42, inventory=500, horizon=20, queue_model="fifo")
obs, _ = env.reset(seed=42)
inv_trajectory = [env.inventory]
queue_ahead_log = []

done = False
while not done:
    obs, r, term, trunc, info = env.step(0.0)  # passive order placement
    inv_trajectory.append(env.inventory)
    queue_ahead_log.append(info.get("queue_ahead", 0))
    done = term or trunc

print(f"Episode completed in {len(queue_ahead_log)} steps.")
print(f"Initial Inventory: {env.inventory0} | Final: {env.inventory} | Executed: {env.inventory0 - env.inventory}")
print(f"Execution VWAP: {env.execution_vwap():.2f} ticks | Market VWAP: {env.market_vwap():.2f} ticks")

Episode completed in 20 steps.
Initial Inventory: 500 | Final: 0 | Executed: 500
Execution VWAP: 14999.03 ticks | Market VWAP: 14998.99 ticks
